# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badsha123508/probable-umbrella/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## My lane as an ML task

My lane is a **scoring task supported by classification**. The goal is to estimate which content pages are most likely to need review and then rank them by review priority.

The underlying prediction can use a classification model to estimate the probability that a page is showing decline or opportunity. Those probabilities can then be used to produce a ranked review queue.

This is useful because the final action is not simply yes/no. A content team has limited time, so it needs to know which pages should be reviewed first.

In [12]:
lane_task_type = "Scoring supported by classification"
lane_goal = "Rank content pages by priority for human review"

print("Task type:", lane_task_type)
print("Goal:", lane_goal)


Task type: Scoring supported by classification
Goal: Rank content pages by priority for human review


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target or proxy

The target for the starter task is a proxy called `is_declining_label`.

It is defined as `trend_direction == "down"`. This label comes from an observed rule applied to the current data window rather than from a future outcome.

I will treat it as a proxy for content decline/opportunity, not as proof that a page will continue to decline or that a refresh will cause recovery. A stronger future version would use features from a prior time window to predict an observed outcome in a later window.



In [13]:
print("Proxy target definition:")
print("is_declining_label = trend_direction == 'down'")


Proxy target definition:
is_declining_label = trend_direction == 'down'


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Success metric

The main success metric is **Precision@50**.

Precision@50 measures how many of the top 50 pages ranked for review are positive according to the defined target. This metric fits the real decision because the content team has limited review capacity and needs the highest-priority pages near the top of the queue.

For comparison, the starter fixed-rule baseline has a Precision@50 of 0.240. A useful ML model should beat that baseline on the same validation setup rather than only producing a high overall accuracy.

In [14]:
baseline_precision_at_50 = 0.240

print("Primary success metric: Precision@50")
print("Starter baseline Precision@50:", baseline_precision_at_50)
print("Success means improving on the baseline under the same validation setup.")


Primary success metric: Precision@50
Starter baseline Precision@50: 0.24
Success means improving on the baseline under the same validation setup.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## The unit of analysis, as a real dataframe

The unit of analysis is a **content page**. Each modeling row represents one pseudonymized content item after the starter data is filtered and deduplicated.

The dataframe contains observable search, content, and engagement signals such as impressions, sessions, content age, CTR, average position, freshness, and other derived measurements. The target is created from `trend_direction`.

No client names, URLs, private queries, or other sensitive raw-origin information are used.

In [15]:
import pandas as pd

# Load the anonymized starter dataset
data_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

# Apply the starter preparation filters
df = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].copy()

# Keep one row per content item
df = df.drop_duplicates(subset=["content_id"]).copy()

# Create the starter proxy target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Dataframe shape:", df.shape)
print("\nOne row represents: one content page")
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())


Dataframe shape: (30000, 45)

One row represents: one content page

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']

First five rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


In [16]:
!git clone https://github.com/badsha123508/probable-umbrella.git

Cloning into 'probable-umbrella'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 138 (delta 50), reused 95 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.85 MiB | 8.01 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [17]:
import os

print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## Why ML beats a fixed rule here

The problem has several interacting signals, so a single fixed rule may miss useful combinations of evidence. A page can have high impressions but poor CTR, strong search visibility but old content, or declining performance while still receiving meaningful demand.

A fixed rule requires manually chosen thresholds and weights. A classification model can learn combinations of observable signals from historical examples and produce a probability that can be used for ranking.

The model is still decision-support rather than a guarantee that refreshing a page will improve its performance. Human review is needed before taking action.

In [18]:
# Check the number and types of observable signals available
feature_columns = [
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "word_count",
    "search_volume",
    "competition",
    "content_type",
    "main_intent",
]

available_features = [
    col for col in feature_columns
    if col in df.columns
]

print("Observable candidate signals available:", len(available_features))
print("\nSignals:")
for col in available_features:
    print("-", col)

print("\nTarget:", "is_declining_label")
print("Rows available for framing:", len(df))


Observable candidate signals available: 11

Signals:
- impressions_90d
- sessions_90d
- content_age_days
- days_since_last_update
- ctr
- avg_position
- word_count
- search_volume
- competition
- content_type
- main_intent

Target: is_declining_label
Rows available for framing: 30000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.